In [149]:
import warnings
warnings.filterwarnings('ignore')

In [150]:
import pandas as pd
import numpy as np
import pyreadr
import joblib
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import plotly.express as px

from tqdm import tqdm

from dash import Dash, html, dcc, dash_table, Input, Output, callback
import dash_bootstrap_components as dbc

In [151]:
pd.set_option('display.max_columns', 100)

In [152]:
dem_uncont = 34
rep_uncont = 31

In [153]:
seat_sims = pyreadr.read_r('model_output/tot_seats_sims.RDS')[None]
seat_sims = seat_sims.rename({None: 'seats'}, axis=1)
seat_sims['seats'] = seat_sims['seats'].map(lambda x: x + dem_uncont)
seat_sims['winner'] = seat_sims['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims

,seats,winner
0,53,Democrats
1,54,Democrats
2,46,Republicans
3,54,Democrats
4,48,Republicans
...,...,...
7995,51,Democrats
7996,52,Democrats
7997,53,Democrats
7998,49,Republicans


In [154]:
sim_counts = seat_sims.groupby(['seats']).count().reset_index().rename({'winner': 'count'}, axis=1)
n_sims = seat_sims.shape[0]
sim_counts['pct'] = sim_counts['count'] / n_sims * 100
sim_counts['winner'] = sim_counts['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims = pd.merge(left=seat_sims.drop(['winner'], axis=1), right=sim_counts, on='seats', how='left')
def get_desc(winner, pct, seats):
    return f'{winner} wins {seats if seats >= 51 else (100-seats)} seats in {pct:.2f}% of simulations'
#seat_sims['desc'] = seat_sims[['winner', 'pct', 'seats']].apply(lambda x: get_desc(x['winner'], x['pct'], x['seats']), axis=1)
sim_counts.head()

,seats,count,pct,winner
0,38,1,0.0125,Republicans
1,39,1,0.0125,Republicans
2,40,2,0.0250,Republicans
3,41,3,0.0375,Republicans
4,42,13,0.1625,Republicans


In [155]:
np.unique(seat_sims['seats']).shape[0]

26

In [156]:
sims_hist = px.histogram(seat_sims, x='seats', nbins=np.unique(seat_sims['seats']).shape[0]*2, color='winner',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},
                         labels={'seats':'Seats won by Democrats', 'winner': 'Winner'}, template='plotly_white')
sims_hist.update_traces(showlegend=False)
sims_hist.add_vline(x=50.5, line_width=1, line_color='black', annotation_text='51 seats required for Dem majority', 
                    annotation_position='top right')
sims_hist

In [157]:
output_ts = pd.read_csv('model_output/output_over_time.csv')
output_ts.head()

,date,y,geo,type
0,2026-09-27,51.348375,US Senate,seats
1,2026-09-27,58.600000,US Senate,chance
2,2026-09-27,3.816193,US Senate,seats_sd
3,2026-09-27,38.771876,AL,y_pred
4,2026-09-27,3.537242,AL,y_pred_sd


In [158]:
chance_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'chance')]
chance_over_time['rep_chance'] = chance_over_time['y'].map(lambda x: 100 - x)
chance_over_time = chance_over_time.rename({'y': 'Democrats', 'rep_chance': 'Republicans'}, axis=1)
chance_time_ser = px.line(chance_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
chance_time_ser.update_traces(hovertemplate="%{y:.1f}%")
chance_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Win Probability (%)',
    title=dict(text="Senate Win Probability Over Time"),
    hovermode="x",
    showlegend=False
)
chance_time_ser

In [159]:
seats_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'seats')]
seats_over_time['rep_seats'] = seats_over_time['y'].map(lambda x: 100 - x)
seats_over_time = seats_over_time.rename({'y': 'Democrats', 'rep_seats': 'Republicans'}, axis=1)
seats_time_ser = px.line(seats_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                        color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
seats_time_ser.update_traces(hovertemplate="%{y:.1f}")
seats_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Average Seats Over All Simulations',
    title=dict(text="Projected Seats Over Time"),
    hovermode="x",
    showlegend=False
)
seats_time_ser

In [160]:
joblib.dump(sims_hist, 'display_data/sims_histogram.pkl')
joblib.dump(chance_time_ser, 'display_data/chance_time_ser.pkl')
joblib.dump(seats_time_ser, 'display_data/seats_time_ser.pkl')

['display_data/seats_time_ser.pkl']

In [161]:
# posterior prediction
post = pyreadr.read_r('model_output/labeled_posterior.RDS')[None]
post_untransp = post.copy()

In [162]:
post = post.T
post.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,7950,7951,7952,7953,7954,7955,7956,7957,7958,7959,7960,7961,7962,7963,7964,7965,7966,7967,7968,7969,7970,7971,7972,7973,7974,7975,7976,7977,7978,7979,7980,7981,7982,7983,7984,7985,7986,7987,7988,7989,7990,7991,7992,7993,7994,7995,7996,7997,7998,7999
AL,-7.940146,-14.023001,-12.207410,-12.259175,-9.936638,-13.777228,-11.797920,-5.641721,-12.716101,-17.696528,-1.459724,-13.631825,-11.461260,-11.941766,-12.522957,-9.637540,-9.928656,-12.937202,-10.331050,-7.832356,-13.227691,-9.139935,-7.326016,-9.678518,-9.378178,-14.561545,-14.142302,-8.501803,-13.363826,-13.206554,-10.689979,-12.256156,-11.902774,-8.401003,-16.436351,-15.436247,-8.264059,-11.193312,-9.077284,-12.942208,-10.900890,-9.220525,-14.340417,-10.931251,-13.156560,-13.656256,-13.271853,-15.496100,-8.031184,-11.863137,...,-11.989225,-11.277749,-10.421968,-15.487729,-7.684978,-11.798185,-13.369041,-8.246947,-12.720357,-8.580913,-16.578993,-16.617335,-10.910955,-7.613108,-17.813476,-7.487406,-16.728028,-3.458135,-12.676258,-8.949761,-11.954868,-10.325853,-4.718504,-21.913226,-3.355096,-14.740453,-12.299583,-13.629419,-8.603743,-12.689298,-11.265364,-12.964615,-11.352265,-8.372205,-17.052398,-12.872705,-11.482682,-19.102732,-10.710270,-10.254542,-11.070656,-14.651631,-10.859067,-10.780266,-12.549654,-13.610488,-15.233400,-10.317615,-13.041047,-12.558317
AK,0.588184,5.741808,-0.003011,5.656088,-0.830248,4.733283,5.152286,5.496214,-0.428130,-5.507872,10.995473,0.471499,-0.980809,6.594455,-0.678811,3.294157,5.811489,4.833505,-0.263729,2.943517,6.940624,4.397037,2.452849,5.192536,4.856206,1.618167,2.565640,3.632060,-0.814348,1.803591,3.615908,5.804263,7.125378,8.408317,2.996042,-0.016422,3.660383,0.250116,-0.897137,3.312547,-1.052277,3.715285,7.081933,4.259616,2.739279,-1.543187,1.055280,3.109775,2.352726,4.439175,...,4.107127,3.057179,10.118745,-0.312354,7.653102,2.883223,1.281812,3.668273,-1.334919,10.564650,2.449003,-0.824853,7.101963,3.367864,3.931372,1.631526,5.058167,2.156696,-0.602821,7.591345,1.684104,3.367467,8.648181,0.168681,4.204359,-2.371606,6.352483,5.114724,3.568829,1.084697,-3.118328,5.438187,-0.714956,6.012182,1.901770,3.079246,3.327223,7.373009,4.092235,4.064109,0.209510,4.364266,3.953903,0.459392,4.162068,0.840925,1.802999,0.649282,3.664209,1.782662
AR,-11.279258,-13.046933,-9.431891,-12.108333,-10.897774,-9.103657,-10.150764,-6.702092,-15.319313,-15.050693,-2.272294,-12.813067,-13.676662,-8.185809,-13.156672,-11.800679,-11.135161,-9.635011,-12.000499,-8.341001,-12.325217,-11.784309,-8.107176,-8.273857,-12.510660,-9.925300,-12.353917,-8.991339,-12.336318,-12.117975,-9.478933,-10.201560,-7.615015,-8.377305,-14.552641,-14.016373,-7.280373,-10.823174,-12.339012,-11.413769,-11.469938,-7.349938,-13.149146,-14.994889,-13.635827,-11.838944,-9.962836,-14.937825,-11.838124,-12.799245,...,-13.843420,-12.463675,-9.098484,-14.449268,-4.837027,-13.303277,-13.676418,-12.408506,-11.259216,-4.966131,-15.600689,-13.185045,-9.206669,-8.560951,-14.500930,-5.689637,-16.466962,-2.163729,-10.295917,-9.564902,-12.833802,-7.350243,-5.879053,-16.489765,-4.342049,-17.473015,-10.135924,-17.152680,-6.277892,-10.751013,-10.158318,-11.876813,-10.604931,-3.463687,-18.523039,-10.632467,-8.667532,-13.417314,-7.037096,-13.094268,-12.144481,-11.702378,-9.597670,-8.135452,-12.300689,-14.331385,-17.978826,-12.256266,-8.696110,-11.778307
CO,11.521371,12.303367,7.376874,14.228098,8.982420,12.666424,14.467985,13.359549,10.933298,7.259313,13.426175,13.913728,11.030134,16.292769,8.043690,15.774473,13.830788,11.916087,6.926954,6.194898,13.053347,18.811558,10.017412,16.552482,13.804646,11.599338,11.425618,20.175681,7.842636,13.639930,14.199110,10.580874,9.497747,10.600649,11.883092,5.929108,16.436881,11.323655,15.541072,9.326586,13.171787,12.150353,11.782571,13.719513,10.716145,11.852686,7.770624,11.235184,12.513134,9.656140,...,13.930485,8.759512,17.532733,1

In [163]:
sim_corr = post_untransp.corr()

In [164]:
sim_corr.head()

,AL,AK,AR,CO,DE,FL,GA,ID,IL,IA,KS,KY,LA,ME,MA,MI,MN,MS,MT,NE,NH,NJ,NM,NC,OH,OK,OR,RI,SC,SD,TN,TX,VA,WV,WY
AL,1.000000,0.516423,0.740060,0.490271,0.497199,0.706546,0.532239,0.737059,0.708323,0.744495,0.738368,0.541054,0.751724,0.736799,0.522739,0.734949,0.708386,0.743801,0.701739,0.725304,0.504085,0.506866,0.508338,0.705147,0.507007,0.758487,0.507685,0.490371,0.491389,0.511815,0.557306,0.701951,0.506536,0.741151,0.738956
AK,0.516423,1.000000,0.516825,0.521094,0.524941,0.500398,0.558389,0.527043,0.503089,0.517994,0.518699,0.566212,0.523894,0.506762,0.560681,0.511373,0.499440,0.517082,0.498395,0.516285,0.560992,0.539124,0.553389,0.498125,0.551955,0.520859,0.547119,0.549544,0.538552,0.541577,0.597650,0.508454,0.544186,0.520549,0.519451
AR,0.740060,0.516825,1.000000,0.487464,0.500004,0.703860,0.528016,0.736206,0.700086,0.741651,0.738009,0.537221,0.742276,0.730403,0.523598,0.729286,0.700436,0.725894,0.693644,0.721852,0.509517,0.500512,0.502831,0.705532,0.504677,0.744325,0.501858,0.496858,0.495346,0.515374,0.557057,0.701945,0.498877,0.735944,0.735098
CO,0.490271,0.521094,0.487464,1.000000,0.708952,0.654456,0.550671,0.497493,0.655188,0.503370,0.487857,0.549427,0.503961,0.484404,0.537941,0.487384,0.642793,0.493852,0.653290,0.477820,0.711333,0.715793,0.720447,0.647543,0.705916,0.499940,0.710488,0.702320,0.698922,0.530681,0.571426,0.656062,0.703960,0.496860,0.494059
DE,0.497199,0.524941,0.500004,0.708952,1.000000,0.667729,0.543314,0.503915,0.665174,0.508833,0.499927,0.547752,0.503504,0.500670,0.548978,0.501749,0.663293,0.499242,0.664185,0.489431,0.719215,0.725936,0.729571,0.661434,0.715800,0.515663,0.710036,0.712681,0.709081,0.521118,0.577942,0.661477,0.719020,0.499426,0.509313


In [165]:
post.shape

(35, 8000)

In [166]:
def get_tipping_point(sim):
    """
    :param sim: Series representing one posterior draw or "simulation"
    :type sim: pd.Series
    """
    seats_won_dem = np.sum(sim > 0)
    if seats_won_dem >= 51 - dem_uncont: # Democrats win House in this draw
        sim = sim.sort_values(ascending=True)
        won_seats = sim[sim > 0]
        seat_margin = seats_won_dem - (51 - dem_uncont)
    else: # Republicans win House in this draw
        sim = sim.sort_values(ascending=False)
        won_seats = sim[sim < 0]
        seat_margin = (100 - seats_won_dem - dem_uncont - rep_uncont) - (50 - rep_uncont)
    return won_seats.iloc[seat_margin - 1], won_seats.index[seat_margin - 1]

In [167]:
tipping_points = np.array([]) # tipping point for *each sim*

for i in tqdm(range(post.shape[1])):
    sim = post.iloc[:, i]
    _, tp_seat = get_tipping_point(sim)
    tipping_points = np.append(tipping_points, tp_seat)

tipping_points

100%|████████████████████████████████████████████████████████████████████████████| 8000/8000 [00:06<00:00, 1172.33it/s]


array(['AK', 'MI', 'TX', ..., 'IA', 'IA', 'AK'],
      shape=(8000,), dtype='<U32')

In [168]:
data = pd.read_csv('../../model_output/senate_predictions.csv')
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.059961,0,0,32.999181,0.619720,53.994266,0.619720,East South Central,2026,0.000000,0.0,0.000000,5,Senate,0.142508,0.619720,0.787223,20.995085,-20.566481,-49.622498,0,-99.244996,38.634103,3.540408,0.0500,1,31.817054,45.701646
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.059961,0,1,48.974227,3.101556,47.234091,3.101556,Pacific,2026,0.000000,0.0,0.000000,5,Senate,9989.631483,3.101556,1.761123,-1.740137,-3.850644,49.948144,-1,99.896288,53.266579,3.405337,83.0750,2,46.588610,60.089740
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.059961,0,1,38.189521,0.391559,49.190029,0.391559,West South Central,2026,0.000000,0.0,0.000000,5,Senate,1718.789084,0.391559,0.625747,11.000508,-21.561082,-8.541719,-1,-17.083438,39.159225,3.568093,0.1250,3,32.057140,46.058828
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30,5941663.80,97.730058,2.269942,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-9.059961,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9551.164309,0.000000,0.000000,0.000000,20.989096,47.730058,1,95.460117,61.875680,3.504022,99.9625,4,55.087281,68.779731
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27,2019499.90,98.524126,1.475874,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-9.059961,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9707.003443,0.000000,0.000000,0.000000,25.082350,48.524126,1,97.048252,63.418907,3.504468,99.9875,5,56.580599,70.375428


In [169]:
data['tipping_point_prob'] = data['state_po'].map(lambda x: np.mean(tipping_points == x) * 100)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.059961,0,0,32.999181,0.619720,53.994266,0.619720,East South Central,2026,0.000000,0.0,0.000000,5,Senate,0.142508,0.619720,0.787223,20.995085,-20.566481,-49.622498,0,-99.244996,38.634103,3.540408,0.0500,1,31.817054,45.701646,0.0000
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.059961,0,1,48.974227,3.101556,47.234091,3.101556,Pacific,2026,0.000000,0.0,0.000000,5,Senate,9989.631483,3.101556,1.761123,-1.740137,-3.850644,49.948144,-1,99.896288,53.266579,3.405337,83.0750,2,46.588610,60.089740,4.8375
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.059961,0,1,38.189521,0.391559,49.190029,0.391559,West South Central,2026,0.000000,0.0,0.000000,5,Senate,1718.789084,0.391559,0.625747,11.000508,-21.561082,-8.541719,-1,-17.083438,39.159225,3.568093,0.1250,3,32.057140,46.058828,0.0000
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30,5941663.80,97.730058,2.269942,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-9.059961,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9551.164309,0.000000,0.000000,0.000000,20.989096,47.730058,1,95.460117,61.875680,3.504022,99.9625,4,55.087281,68.779731,0.1500
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27,2019499.90,98.524126,1.475874,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-9.059961,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9707.003443,0.000000,0.000000,0.000000,25.082350,48.524126,1,97.048252,63.418907,3.504468,99.9875,5,56.580599,70.375428,0.7125


In [170]:
data.sort_values('tipping_point_prob', ascending=False).head(5)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
15,15,Michigan,Abdul El-Sayed,Mike J. Rogers,False,False,D,MI,"EL-SAYED, ABDUL","ROGERS, MICHAEL J",14408058.06,6831818.66,21239876.72,67.834942,32.165058,-0.192456,49.278763,77.709102,2.333008,4.148895,12.968600,0.319413,32.445883,7404258,2673073,10077331,73.474395,26.525605,0.987401,-9.059961,0,0,47.859332,8.288265,44.956912,8.288265,East North Central,2026,0.0,0.00000,0.00000,2,Senate,4601.579360,8.288265,2.878935,-2.902420,8.675048,17.834942,0,35.669884,50.829342,3.538004,59.1125,16,44.051944,57.825866,11.4500
31,31,Texas,James Talarico,Ken Paxton,False,False,R,TX,"TALARICO, JAMES","PAXTON, WARREN KENNETH JR.",65585741.88,9020177.98,74605919.86,87.909568,12.090432,-5.916874,43.060966,48.286718,4.530331,31.642337,13.173480,0.180615,33.823334,24400697,4744808,29145505,83.720275,16.279725,0.987401,-9.059961,0,0,48.330668,7.245563,45.576386,7.245563,West South Central,2026,0.0,0.25284,-0.25284,6,Senate,7728.092153,7.245563,2.691758,-2.754282,-2.773787,37.909568,0,75.819136,51.127108,3.629425,62.1000,32,44.041891,58.333230,11.3250
9,9,Iowa,Josh Turek,Ashley Hinson,False,False,R,IA,"TUREK, JOSHUA","ARENHOLZ, ASHLEY HINSON",5835816.67,5267628.55,11103445.22,52.558612,47.441388,-6.092833,43.277201,88.847514,1.696738,4.401391,3.028533,0.168574,31.423453,2014831,1175538,3190369,63.153541,36.846459,0.987401,-9.059961,0,0,46.647197,5.791369,46.135733,5.791369,West North Central,2026,0.0,0.00000,0.00000,3,Senate,2762.407675,5.791369,2.406526,-0.511465,-3.125706,2.558612,0,5.117224,49.735464,3.515463,46.0125,10,42.742452,56.613552,9.6625
24,24,Ohio,Sherrod Brown,Jon Husted,False,True,R,OH,"BROWN, SHERROD","HUSTED, JON",31146133.40,6280012.05,37426145.45,83.220254,16.779746,-5.266554,44.343385,80.925356,1.708882,3.291326,11.486903,0.076606,31.487184,9001099,2798349,11799448,76.284069,23.715931,0.987401,-9.059961,0,1,48.768743,2.677890,44.102599,2.677890,East North Central,2026,0.0,0.00000,0.00000,2,Senate,6925.610718,2.677890,1.636426,-4.666143,-1.473147,33.220254,-1,66.440509,51.136400,3.466911,63.1875,25,44.223027,57.989118,8.9000
28,28,South Carolina,Annie Andrews,Darline Graham,False,False,R,SC,"ANDREWS, ANNIE","GRAHAM, DARLINE",10677209.93,299782.64,10976992.57,97.268991,2.731009,-8.282832,40.938317,67.144996,1.378347,4.124393,24.827438,0.196629,32.129153,3477869,1640556,5118425,67.948031,32.051969,0.987401,-9.059961,0,0,44.596019,1.612311,45.788282,1.612311,South Atlantic,2026,0.0,0.00000,0.00000,5,Senate,9461.256637,1.612311,1.269768,1.192263,-7.505704,47.268991,0,94.537982,49.495624,3.547904,44.1500,29,42.487353,56.440987,8.3500


In [171]:
def get_rating(dem_chance):
    if dem_chance > 100:
        raise ValueError('Invalid win chance.')
    if dem_chance > 95:
        return 'Safe D'
    elif dem_chance >= 90:
        return 'Very Likely D'
    elif dem_chance >= 75:
        return 'Likely D'
    elif dem_chance >= 65:
        return 'Lean D'
    elif dem_chance >= 60:
        return 'Tilt D'
    elif dem_chance >= 40:
        return 'Tossup'
    elif dem_chance >= 35:
        return 'Tilt R'
    elif dem_chance >= 25:
        return 'Lean R'
    elif dem_chance >= 10:
        return 'Likely R'
    elif dem_chance >= 5:
        return 'Very Likely R'
    else:
        return 'Safe R'

def get_matchup(dem_cand, rep_cand, dem_inc_any, rep_inc_any):
    indie_d = ['Dan Osborn', 'Brian Bengs', 'Todd Achilles', 'Seth Bodnar']
    indie_r = []
    
    if dem_cand in indie_d:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (I)'
        dem_color = '#792ba6'
    else:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (D)'
        dem_color = '#366bbf'
    
    if rep_cand in indie_r:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (I)'
        rep_color = '#792ba6'
    else:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (R)'
        rep_color = '#e63929'

    return f'<b style="color:{dem_color};">' + dem_lab + f'</b> vs <b style="color:{rep_color};">' + rep_lab + '</b>'

In [172]:
data['rating'] = data['chance'].map(lambda x: get_rating(x))
for party in ['dem', 'rep']:
    data[f'{party}_cand'] = data[f'{party}_cand'].map(lambda x: 'TBD' if x[:3] == 'TBD' else x)
data['matchup'] = data[['dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand'],
                                                                                                          x['dem_inc_any'], x['rep_inc_any']), axis=1)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.059961,0,0,32.999181,0.619720,53.994266,0.619720,East South Central,2026,0.000000,0.0,0.000000,5,Senate,0.142508,0.619720,0.787223,20.995085,-20.566481,-49.622498,0,-99.244996,38.634103,3.540408,0.0500,1,31.817054,45.701646,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b..."
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.059961,0,1,48.974227,3.101556,47.234091,3.101556,Pacific,2026,0.000000,0.0,0.000000,5,Senate,9989.631483,3.101556,1.761123,-1.740137,-3.850644,49.948144,-1,99.896288,53.266579,3.405337,83.0750,2,46.588610,60.089740,4.8375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>..."
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.059961,0,1,38.189521,0.391559,49.190029,0.391559,West South Central,2026,0.000000,0.0,0.000000,5,Senate,1718.789084,0.391559,0.625747,11.000508,-21.561082,-8.541719,-1,-17.083438,39.159225,3.568093,0.1250,3,32.057140,46.058828,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<..."
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30,5941663.80,97.730058,2.269942,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-9.059961,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9551.164309,0.000000,0.000000,0.000000,20.989096,47.730058,1,95.460117,61.875680,3.504022,99.9625,4,55.087281,68.779731,0.1500,Safe D,"<b style=""color:#366bbf;"">John Hickenlooper* (..."
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27,2019499.90,98.524126,1.475874,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-9.059961,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9707.003443,0.000000,0.000000,0.000000,25.082350,48.524126,1,97.048252,63.418907,3.504468,99.9875,5,56.580599,70.375428,0.7125,Safe D,"<b style=""color:#366bbf;"">Chris Coons* (D)</b>..."


In [173]:
data['projected_winner'] = data['chance'].map(lambda x: 'D' if x > 50 else 'R')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.059961,0,0,32.999181,0.619720,53.994266,0.619720,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.619720,0.787223,20.995085,-20.566481,-49.622498,0,-99.244996,38.634103,3.540408,0.050,1,31.817054,45.701646,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.059961,0,1,48.974227,3.101556,47.234091,3.101556,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,3.101556,1.761123,-1.740137,-3.850644,49.948144,-1,99.896288,53.266579,3.405337,83.075,2,46.588610,60.089740,4.8375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.059961,0,1,38.189521,0.391559,49.190029,0.391559,West South Central,2026,0.0,0.0,0.0,5,Senate,1718.789084,0.391559,0.625747,11.000508,-21.561082,-8.541719,-1,-17.083438,39.159225,3.568093,0.125,3,32.057140,46.058828,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R


In [174]:
data['hold'] = data['projected_winner'] ==  data['current_party']
data['flip'] = data['hold'].map(lambda x: not x)
data['flip_indic'] = data['flip'].map(lambda x: 'Flip' if x else '')
#data['flip'] = data['flip'].map(lambda x: 'Yes' if x else 'No')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.059961,0,0,32.999181,0.619720,53.994266,0.619720,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.619720,0.787223,20.995085,-20.566481,-49.622498,0,-99.244996,38.634103,3.540408,0.050,1,31.817054,45.701646,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.059961,0,1,48.974227,3.101556,47.234091,3.101556,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,3.101556,1.761123,-1.740137,-3.850644,49.948144,-1,99.896288,53.266579,3.405337,83.075,2,46.588610,60.089740,4.8375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip


In [175]:
data['projected_2p_margin'] = data['y_pred'].map(lambda y_pred: f'D+{y_pred - (100-y_pred):.1f}' if y_pred > 50 else f'R+{(100-y_pred) - y_pred:.1f}')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.059961,0,0,32.999181,0.619720,53.994266,0.619720,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.619720,0.787223,20.995085,-20.566481,-49.622498,0,-99.244996,38.634103,3.540408,0.050,1,31.817054,45.701646,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+22.7
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.059961,0,1,48.974227,3.101556,47.234091,3.101556,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,3.101556,1.761123,-1.740137,-3.850644,49.948144,-1,99.896288,53.266579,3.405337,83.075,2,46.588610,60.089740,4.8375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.5
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.059961,0,1,38.189521,0.391559,49.190029,0.391559,West South Central,2026,0.0,0.0,0.0,5,Senate,1718.789084,0.391559,0.625747,11.000508,-21.561082,-8.541719,-1,-17.083438,39.159225,3.568093,0.125,3,32.057140,46.058828,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+21.7


In [176]:
data['rep_chance'] = data['chance'].map(lambda x: 100 - x)
data['rounded_dem_chance'] = data['chance'].map(lambda x: np.round(x, 1))
data['rounded_rep_chance'] = data['rep_chance'].map(lambda x: np.round(x, 1))
data['disp_dem_chance'] = data['rounded_dem_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data['disp_rep_chance'] = data['rounded_rep_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.059961,0,0,32.999181,0.619720,53.994266,0.619720,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.619720,0.787223,20.995085,-20.566481,-49.622498,0,-99.244996,38.634103,3.540408,0.050,1,31.817054,45.701646,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+22.7,99.950,0.0,100.0,<1%,>99%
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.059961,0,1,48.974227,3.101556,47.234091,3.101556,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,3.101556,1.761123,-1.740137,-3.850644,49.948144,-1,99.896288,53.266579,3.405337,83.075,2,46.588610,60.089740,4.8375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.5,16.925,83.1,16.9,83.1%,16.9%
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.059961,0,1,38.189521,0.391559,49.190029,0.391559,West South Central,2026,0.0,0.0,0.0,5,Senate,1718.789084,0.391559,0.625747,11.000508,-21.561082,-8.541719,-1,-17.083438,39.159225,3.568093,0.125,3,32.057140,46.058828,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+21.7,99.875,0.1,99.9,<1%,>99%


In [177]:
data['swing_24_to_26'] = data['y_pred'].astype(float).map(lambda x: x - (100 - x)) - data['dem_2p_24'].astype(float).map(lambda x: x - (100 - x))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.059961,0,0,32.999181,0.619720,53.994266,0.619720,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.619720,0.787223,20.995085,-20.566481,-49.622498,0,-99.244996,38.634103,3.540408,0.050,1,31.817054,45.701646,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+22.7,99.950,0.0,100.0,<1%,>99%,8.149430
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.059961,0,1,48.974227,3.101556,47.234091,3.101556,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,3.101556,1.761123,-1.740137,-3.850644,49.948144,-1,99.896288,53.266579,3.405337,83.075,2,46.588610,60.089740,4.8375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.5,16.925,83.1,16.9,83.1%,16.9%,20.226284
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.059961,0,1,38.189521,0.391559,49.190029,0.391559,West South Central,2026,0.0,0.0,0.0,5,Senate,1718.789084,0.391559,0.625747,11.000508,-21.561082,-8.541719,-1,-17.083438,39.159225,3.568093,0.125,3,32.057140,46.058828,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+21.7,99.875,0.1,99.9,<1%,>99%,9.658436


In [178]:
data['disp_24_to_26_swing'] = data['swing_24_to_26'].map(lambda x: f'D+{x:.1f}' if x > 0 else f'R+{abs(x):.1f}')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26,disp_24_to_26_swing
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.059961,0,0,32.999181,0.619720,53.994266,0.619720,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.619720,0.787223,20.995085,-20.566481,-49.622498,0,-99.244996,38.634103,3.540408,0.050,1,31.817054,45.701646,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+22.7,99.950,0.0,100.0,<1%,>99%,8.149430,D+8.1
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.059961,0,1,48.974227,3.101556,47.234091,3.101556,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,3.101556,1.761123,-1.740137,-3.850644,49.948144,-1,99.896288,53.266579,3.405337,83.075,2,46.588610,60.089740,4.8375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.5,16.925,83.1,16.9,83.1%,16.9%,20.226284,D+20.2


In [179]:
tab_data = data[['state', 'dem_cand', 'rep_cand', 'rating', 'disp_dem_chance', 'disp_rep_chance', 'projected_2p_margin', 'disp_24_to_26_swing',
                 'tipping_point_prob']]
for party in ['dem', 'rep']:
    tab_data[f'disp_{party}_chance'] = tab_data[f'disp_{party}_chance'].map(lambda x: f'<p style="color:{'blue' if party == 'dem' else 'red'};">{x}</p>')
    tab_data[f'{party}_cand'] = tab_data[f'{party}_cand'].map(lambda x: f'{x} (Ind)' if x in ['Dan Osborn', 'Brian Bengs', 
                                                                                              'Todd Achilles', 'Seth Bodnar'] else x)
tab_data = tab_data.rename({
    'state': 'State',
    'dem_cand': 'Democrat',
    'rep_cand': 'Republican',
    'rating': 'Rating',
    'disp_dem_chance': 'Dem Chance',
    'disp_rep_chance': 'Rep Chance',
    'projected_2p_margin': 'Projected Margin',
    'tipping_point_prob': 'Tipping Point Chance',
    'disp_24_to_26_swing': 'Swing from 2024 Pres'
}, axis=1)
tab_data.head()

,State,Democrat,Republican,Rating,Dem Chance,Rep Chance,Projected Margin,Swing from 2024 Pres,Tipping Point Chance
0,Alabama,Everett Weiss,Barry Moore,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+22.7,D+8.1,0.0000
1,Alaska,Mary Peltola,Dan S. Sullivan,Likely D,"<p style=""color:blue;"">83.1%</p>","<p style=""color:red;"">16.9%</p>",D+6.5,D+20.2,4.8375
2,Arkansas,Hallie Shoffner,Tom Cotton,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+21.7,D+9.7,0.0000
3,Colorado,John Hickenlooper,Mark Baisley,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+23.8,D+12.5,0.1500
4,Delaware,Chris Coons,Michael Katz,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+26.8,D+11.9,0.7125


In [180]:
data['rep_pred'] = data['y_pred'].map(lambda x: 100 - x)
data['projected_2p_margin_number'] = data['rep_pred'] - data['y_pred']
data['proj_seat_lean'] = data['projected_2p_margin_number'] - data['generic_ballot_avg']
sv_bias = float(data.sort_values('tipping_point_prob', ascending=False).reset_index().loc[0, 'proj_seat_lean']) # Seats-votes bias, + = R, - = D
sv_bias

7.401276752864749

In [181]:
topline_stats = pd.read_csv('display_data/topline_stats.csv')
if 'sv_bias' in topline_stats['vars'].values:
    topline_stats = topline_stats[topline_stats['vars'] != 'sv_bias']
topline_stats = pd.concat([topline_stats, pd.DataFrame({'vars': ['sv_bias'], 'x': sv_bias})], axis=0)
topline_stats.to_csv('display_data/topline_stats.csv')
topline_stats

,Unnamed: 0,vars,x
0,0.0,means_seats_tot,51.465875
1,1.0,chamber_win_chance,59.212500
0,NaN,sv_bias,7.401277


In [182]:
mean_seats_tot = topline_stats[topline_stats['vars'] == 'means_seats_tot']['x'].values[0]
chamber_win_chance = topline_stats[topline_stats['vars'] == 'chamber_win_chance']['x'].values[0]

In [183]:
chances = topline_stats[topline_stats['vars'] == 'chamber_win_chance'].set_index(['vars']).T
chances['Republicans'] = chances['chamber_win_chance'].map(lambda x: 100 - x)
chances = chances.rename({'chamber_win_chance': 'Democrats'}, axis=1).T.reset_index()
chances = chances.rename({'x': 'Win Probability'}, axis=1)
seats = topline_stats[topline_stats['vars'] == 'means_seats_tot'].set_index(['vars']).T
seats['Republicans'] = seats['means_seats_tot'].map(lambda x: 100 - x)
seats = seats.rename({'means_seats_tot': 'Democrats'}, axis=1).T.reset_index().rename({'x': 'Seat Share'}, axis=1)
summary_stats = pd.merge(left=chances, right=seats, on='vars', how='inner').rename({'vars': 'Party'}, axis=1)
summary_stats.to_csv('display_data/summary_stats.csv')
summary_stats

,Party,Unnamed: 0_x,Win Probability,Unnamed: 0_y,Seat Share
0,Democrats,1.0,59.2125,0.0,51.465875
1,Republicans,99.0,40.7875,100.0,48.534125


In [184]:
incl_cols = ['state', 'dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any', 'rating', 'chance', 'disp_dem_chance',
            'disp_rep_chance', 'projected_2p_margin', 'y_pred', 'flip_indic', 'matchup', 'swing_24_to_26', 'disp_24_to_26_swing']
disp_data = data[incl_cols]

In [185]:
rating_ts = output_ts[output_ts['type'] == 'chance']
rating_ts['rating'] = rating_ts['y'].map(get_rating)
rating_ts['date'] = pd.to_datetime(rating_ts['date'])
rating_ts.head()

,date,y,geo,type,rating
1,2026-09-27,58.6000,US Senate,chance,Tossup
5,2026-09-27,0.0500,AL,chance,Safe R
10,2026-09-27,79.8875,AK,chance,Likely D
15,2026-09-27,0.1250,AR,chance,Safe R
20,2026-09-27,99.9625,CO,chance,Safe D


In [186]:
## Alert when there's a rating change
today_date = rating_ts['date'].max()
yesterday = rating_ts[rating_ts['date'] != today_date]['date'].max()

today_ratings = rating_ts[rating_ts['date'] == today_date]
yester_ratings = rating_ts[rating_ts['date'] == yesterday]
compare_df = pd.merge(left=today_ratings, right=yester_ratings, on='geo', how='left', suffixes=['_t', '_y'])
compare_df.head()

,date_t,y_t,geo,type_t,rating_t,date_y,y_y,type_y,rating_y
0,2026-10-03,59.2125,US Senate,chance,Tossup,2026-10-02,58.6375,chance,Tossup
1,2026-10-03,0.0500,AL,chance,Safe R,2026-10-02,0.0500,chance,Safe R
2,2026-10-03,83.0750,AK,chance,Likely D,2026-10-02,80.3000,chance,Likely D
3,2026-10-03,0.1250,AR,chance,Safe R,2026-10-02,0.1250,chance,Safe R
4,2026-10-03,99.9625,CO,chance,Safe D,2026-10-02,99.9625,chance,Safe D


In [187]:
## This is the cell that actually alerts when there's a rating change
if any(compare_df['rating_t'] != compare_df['rating_y']):
    changes_df = compare_df[compare_df['rating_t'] != compare_df['rating_y']][['geo', 'rating_y', 'rating_t']]
    print(changes_df)
else:
    print("No rating changes today!")

No rating changes today!


In [188]:
if any(compare_df['rating_t'] != compare_df['rating_y']):
    print(changes_df.shape)

In [189]:
disp_data.to_csv('display_data/choropleth_display_data.csv')
tab_data.to_csv('display_data/table_display_data.csv')
data.to_csv('display_data/all_data.csv')